### ロジスティック回帰とは
- あるクラスに属する確率を予測する手法（分類問題に使用）
- 出力を 0から1の確率に収める必要があり、シグモイド関数を使用。

||内容|
|----|----|
|出力|クラス1に属する確率|
|損失関数|交差エントロピー|
|最適化|勾配降下法|
|予測|確率≥0.5 ならクラス1|

In [1]:
# Irisデータセットを使用。
# 2クラス分類にするため、品種0（setosa）と品種1（versicolor）のみを使用。

import numpy as np
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# データの読み込み
iris = load_iris()
X = iris.data[iris.target != 2]
y = iris.target[iris.target != 2]

# 訓練データとテストデータに分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# モデルの学習
model = LogisticRegression()
model.fit(X_train, y_train)

# 予測
y_pred = model.predict(X_test)

print(f"正解率: {accuracy_score(y_test, y_pred):.4f}")

正解率: 1.0000


In [2]:
# ロジスティック回帰をscratchで実装
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

class LogisticRegression:
    def __init__(self, lr=0.1, n_epochs=1000):
        self.lr = lr
        self.n_epochs = n_epochs
        self.w = None
        self.b = None

    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0.0

        for _ in range(self.n_epochs):
            z = X @ self.w + self.b
            y_hat = self._sigmoid(z)

            # 勾配の計算
            error = y_hat - y
            dw = (X.T @ error) / n_samples
            db = np.mean(error)

            # パラメータの更新
            self.w -= self.lr * dw
            self.b -= self.lr * db

    def predict_proba(self, X):
        z = X @ self.w + self.b
        return self._sigmoid(z)

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)


# 実行
iris = load_iris()
X = iris.data[iris.target != 2]
y = iris.target[iris.target != 2]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(lr=0.1, n_epochs=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"正解率: {accuracy_score(y_test, y_pred):.4f}")

正解率: 1.0000


`_sigmoid` メソッド

シグモイド関数をそのまま実装しています。アンスコをつけているのは、クラス内部でのみ使うプライベートメソッドであることを示すPythonの慣例です。

`fit` メソッドの勾配計算

X.T @ error は ∑i e_i*x_i の行列演算で、ループを使わずにまとめて計算することができます。導出で求めた式と見比べると、そのまま対応していることがわかります。

`predict` と `predict_proba` の分離

sklearnに倣って、確率を返すメソッドと0 or 1ラベルを返すメソッドを分けています。閾値を変えたい場合に対応しやすくなります。